<a href="https://colab.research.google.com/github/yuri-maradini/TempSal/blob/main/src/train_ueyes_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning TempSAL su UEyes — Colab

Notebook pronto per lanciare il training vero (Step 4) su GPU, invece che sulla CPU locale.

**Prima di eseguire questo notebook**, su Google Drive crea una cartella (default atteso: `MyDrive/TempSAL_UEyes/`) contenente:
- `multilevel_tempsal.pt` — il checkpoint pre-addestrato originale
- `data_ueyes.zip` — l'archivio di `data_ueyes/` generato in locale (Step 1-2)

Poi: **Runtime → Cambia tipo di runtime → GPU**, prima di eseguire le celle.

In [13]:
# Controllo che sia stata assegnata una GPU
!nvidia-smi

Fri Sep  4 15:03:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os

# Cambia questo path se hai usato un nome/percorso diverso su Drive
DRIVE_DIR = '/content/drive/MyDrive/TempSAL_UEyes'

assert os.path.isdir(DRIVE_DIR), (
    f"Cartella non trovata: {DRIVE_DIR}\n"
    "Creala su Drive e caricaci multilevel_tempsal.pt + data_ueyes.zip prima di continuare."
)
print('Contenuto trovato su Drive:', os.listdir(DRIVE_DIR))

Contenuto trovato su Drive: ['multilevel_tempsal.pt', 'data_ueyes.zip', 'multilevel_tempsal_ueyes.pt', 'multilevel_tempsal_ueyes_v2.pt']


In [16]:
# Codice: sempre aggiornato da GitHub, non serve preparazione
!git clone https://github.com/yuri-maradini/TempSal.git /content/TempSal

fatal: destination path '/content/TempSal' already exists and is not an empty directory.


In [17]:
import shutil

os.makedirs('/content/TempSal/src/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal.pt',
)
# multilevel_tempsal_ueyes_v2.pt e' il checkpoint della seconda run (20 epoche,
# epoca 18): questa run riparte da li' -- la testa temporale e' gia' ben
# adattata, l'esperimento isola l'effetto di sbloccare anche il backbone.
shutil.copy(
    f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v2.pt',
    '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v2.pt',
)
print('Checkpoint copiati (originale + v2, il warm-start di questa run).')

Checkpoint copiati (originale + v2, il warm-start di questa run).


In [18]:
import time
import zipfile

# Estratto sul disco locale di Colab (veloce), non lasciato sul mount di Drive
# (l'I/O su Drive montato e' molto piu' lento per tanti file piccoli, e qui
# ce ne sono migliaia tra immagini, mappe e volumi temporali).
t0 = time.time()
with zipfile.ZipFile(f'{DRIVE_DIR}/data_ueyes.zip') as zf:
    zf.extractall('/content/TempSal/')
print(f'Dati estratti in {time.time() - t0:.0f}s')

Dati estratti in 16s


In [19]:
# Controllo veloce di integrita': i conteggi devono combaciare con quelli
# verificati in locale (1872 train / 108 val per ciascuna sottocartella)
for sub in ['images', 'maps', 'fixation_maps', 'saliency_volumes_5', 'fixation_volumes_5']:
    for split in ['train', 'val']:
        d = f'/content/TempSal/data_ueyes/{sub}/{split}'
        n = len(os.listdir(d)) if os.path.isdir(d) else 'MANCANTE'
        print(f'{sub:22s} {split:5s} -> {n}')

images                 train -> 1872
images                 val   -> 108
maps                   train -> 1872
maps                   val   -> 108
fixation_maps          train -> 1872
fixation_maps          val   -> 108
saliency_volumes_5     train -> 9360
saliency_volumes_5     val   -> 540
fixation_volumes_5     train -> 9360
fixation_volumes_5     val   -> 540


In [20]:
# Colab ha gia' PyTorch con supporto CUDA preinstallato: installiamo solo le
# altre dipendenze del progetto, senza toccare torch/torchvision/torchaudio
# (forzare i pin usati in locale, pensati per una build CPU, rischierebbe di
# rimpiazzare la build CUDA gia' pronta di Colab con una incompatibile).
!pip install -q wandb pycocotools ftfy einops clip-anytorch kornia regex

In [21]:
# Verifica che l'installazione sopra non abbia rovinato il supporto CUDA di torch
import torch
print('torch', torch.__version__, '| CUDA disponibile:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU non disponibile: controlla Runtime > Cambia tipo di runtime > GPU'

torch 2.11.0+cu128 | CUDA disponibile: True


## wandb (consigliato per questa run)

Il primo run (10 epoche) è stato fatto con `WANDB_MODE=disabled`: nessuna curva salvata, i numeri per epoca sono stati recuperati a mano dall'output della cella di training. Per questa run vale la pena accendere il logging vero, così le curve di CC/KLDIV/NSS/SIM (aggregate e per-slice) restano disponibili per il confronto e per la tesi senza dover rileggere l'output della cella.

Esegui la cella sotto (chiede l'API key, la trovi su wandb.ai/authorize) prima di lanciare il training. Se preferisci comunque saltarlo, aggiungi di nuovo `WANDB_MODE=disabled` (o `=offline` per salvare i log in locale senza account) davanti al comando `python train.py` nella cella di training.</cell id="HNcrR_LSgcXM">


In [22]:
import wandb
wandb.login()

True

## Training (run 4 — backbone sbloccato, statistiche BatchNorm congelate)

Le prime due run (10 e 20 epoche, solo testa di `pnas_vol` + layer di mixing allenabili, backbone congelato) hanno raggiunto un plateau: sia la mappa aggregata (~epoca 10) sia il ramo temporale (~epoca 16-18) si sono stabilizzati, e la loss di training continuava a scendere mentre le metriche di validazione restavano piatte — segno che il collo di bottiglia era la capacità allenabile del modello, non il numero di epoche.

**Run 3 (sblocco naive del backbone, vedi TODO.md Step 4.5)**: sbloccando anche il backbone (`--train_enc 1`), la curva wandb mostra un calo netto già alla prima epoca (CC 0.718 di `v2` → 0.695 all'epoca 0), seguito da un recupero lento e monotono per tutte le 10 epoche senza mai risalire al livello di `v2`. Causa identificata: `--train_enc 1` mette il backbone in modalità "allenamento" anche per le sue statistiche interne di BatchNorm (`running_mean`/`running_var`), che quindi iniziano a spostarsi verso UEyes fin dal primo forward pass — prima ancora che i pesi imparassero qualcosa di utile. Il run non è stato abbastanza lungo da dimostrare se il backbone sbloccato avrebbe superato `v2` col tempo o no: risultato inconcludente, non un fallimento della strategia in sé.

**Questa run isola l'effetto**: nuova funzione in `train.py` (`freeze_batchnorm_stats`) che forza ogni layer BatchNorm del backbone in modalità `.eval()` quando `--train_enc 1`, così le sue statistiche restano fisse su quelle di `v2` mentre i pesi (comprese le componenti allenabili proprie dei BatchNorm, `weight`/`bias`) continuano ad aggiornarsi normalmente via gradiente — nessun flag nuovo da riga di comando, il comando è identico a quello della run 3, cambia solo il comportamento interno del codice. Verificato in locale su CPU (mini-dataset, `--train_enc 1`): le statistiche del backbone restano **0/603 invariate**, i suoi pesi cambiano **775/777**, esattamente il comportamento voluto.

Il resto è invariato dalla run 3:
- Warm-start da `multilevel_tempsal_ueyes_v2.pt` (non da `v3`, che era peggiore di `v2`).
- `--train_enc 1`, `--lr 1e-6`, `--batch_size 16 --grad_accum_steps 2` (stessa memoria/batch effettivo della run 3, stesso fix OOM).
- `--no_epochs 10`.
- `--model_val_path` → `multilevel_tempsal_ueyes_v4.pt`.

Da controllare dopo la run: se l'epoca 0 parte già vicino al livello di `v2` (conferma che il fix ha eliminato il contraccolpo iniziale) e se il modello supera `v2` entro le 10 epoche.

In [ ]:
%cd /content/TempSal/src
# PYTORCH_CUDA_ALLOC_CONF: riduce la frammentazione dell'allocatore, utile
# ora che la memoria e' molto piu' vicina al limite della T4 (vedi cella
# markdown sopra sull'OOM del primo tentativo).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --enc_model pnas_boosted_multi \
  --dataset_dir ../data_ueyes/ \
  --model_path ./checkpoints/multilevel_tempsal_ueyes_v2.pt \
  --model_vol_path ./checkpoints/multilevel_tempsal_ueyes_v2.pt \
  --train_model 1 \
  --train_enc 1 \
  --lr 1e-6 \
  --batch_size 16 \
  --grad_accum_steps 2 \
  --no_epochs 10 \
  --model_val_path ./checkpoints/multilevel_tempsal_ueyes_v4.pt

In [ ]:
# Copia il checkpoint fine-tuned su Drive, cosi' sopravvive alla chiusura
# della sessione Colab. Puoi rieseguire questa cella anche a training ancora
# in corso, per avere un backup intermedio.
import shutil

src_ckpt = '/content/TempSal/src/checkpoints/multilevel_tempsal_ueyes_v4.pt'
dst_ckpt = f'{DRIVE_DIR}/multilevel_tempsal_ueyes_v4.pt'
shutil.copy(src_ckpt, dst_ckpt)
print('Copiato su Drive:', dst_ckpt)